# PinSAGE — Graph Convolutional Neural Networks for Web-Scale Recommender Systems (2018)

_Papers · Graph Neural Networks_

**Make a Graph Convolutional Network scale to billions of items: pick each node's neighborhood with short random walks, aggregate neighbors as a weighted mean using their random-walk visit counts as importance weights, and train the embeddings with a max-margin ranking loss on (query, positive, negative) item triples.**

---

This notebook is a practice scaffold for the **PinSAGE — Graph Convolutional Neural Networks for Web-Scale Recommender Systems (2018)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random

np.random.seed(0); random.seed(0); torch.manual_seed(0)

# --- 0. Worked example: importance pooling of a tiny neighborhood. ---
# Three neighbors, transformed messages m1=(2,0), m2=(0,4), m3=(1,1); random-walk visit counts 6,3,1.
H = np.array([[2., 0.], [0., 4.], [1., 1.]])
visits = np.array([6., 3., 1.])
alpha = visits / visits.sum()                 # L1-normalized visit counts = importance weights
n_imp = (alpha[:, None] * H).sum(0)           # importance pooling = weighted mean
n_uni = H.mean(0)                              # uniform mean (the ablation)
print("alpha (L1-normalized visits):", np.round(alpha, 4))   # [0.6 0.3 0.1]
print("importance-pooled n_u:", np.round(n_imp, 4))          # [1.3 1.3]
print("uniform-mean n_u:      ", np.round(n_uni, 4))          # [1.     1.6667]

# --- 1. A tiny synthetic item graph: 3 clusters of related items + misleading cross-cluster edges. ---
C, PER = 3, 7; N = C * PER                      # 21 items, 3 clusters of 7
cluster = np.repeat(np.arange(C), PER)
adj = [[] for _ in range(N)]
for i in range(N):
    same  = [j for j in range(N) if j != i and cluster[j] == cluster[i]]
    other = [j for j in range(N) if cluster[j] != cluster[i]]
    adj[i] = same + random.sample(other, 5)    # 5 misleading cross-cluster edges per item

# --- 2. Random-walk importance neighborhoods: visit counts -> top-T -> L1-normalized weights. ---
def visit_counts(start, n_walks=300, length=6):
    cnt = np.zeros(N)
    for _ in range(n_walks):
        cur = start
        for _ in range(length):
            cur = random.choice(adj[cur]); cnt[cur] += 1
    return cnt

T = 6
NB, AL = [], []                                 # PinSAGE: top-T neighbors + importance weights
for u in range(N):
    cnt = visit_counts(u)
    order = np.argsort(-cnt)[:T]                 # top-T most-visited (approx. Personalized PageRank)
    w = cnt[order].astype(np.float32); w = w / w.sum()   # L1-normalize -> alpha
    NB.append(torch.tensor(order)); AL.append(torch.tensor(w))

# node input features, slightly cluster-correlated
feat = torch.randn(N, 8)
for c in range(C): feat[cluster == c] += torch.randn(8) * 1.0

# --- 3. The importance-pooling convolution (Algorithm 1). ---
class PinSageConv(nn.Module):
    def __init__(self, din=8, dh=16, dout=16):
        super().__init__()
        self.Q = nn.Linear(din, dh)             # per-neighbor transform  ReLU(Q h_v + q)
        self.W = nn.Linear(din + dh, dout)      # combine  ReLU(W concat(z_u, n_u) + w)
    def forward(self, z, NB, AL, importance=True):
        out = []
        for u in range(z.shape[0]):
            msg = F.relu(self.Q(z[NB[u]]))                       # transform each neighbor
            n_u = (AL[u][:, None] * msg).sum(0) if importance else msg.mean(0)  # weighted vs uniform pooling
            h = F.relu(self.W(torch.cat([z[u], n_u])))           # combine with self
            out.append(h / (h.norm() + 1e-9))                    # L2-normalize
        return torch.stack(out)

# --- 4. Max-margin ranking loss on (query, positive, negative) triples (Eqn. 1). ---
def hit_rate(z):
    z = F.normalize(z, dim=1); sim = z @ z.t(); sim.fill_diagonal_(-9)
    return float(np.mean([cluster[sim[u].argmax().item()] == cluster[u] for u in range(z.shape[0])]))
def gap(z):
    z = F.normalize(z, dim=1); ia, ie = [], []
    for i in range(z.shape[0]):
        for j in range(z.shape[0]):
            if i != j: (ia if cluster[i] == cluster[j] else ie).append(float((z[i] * z[j]).sum()))
    return float(np.mean(ia)), float(np.mean(ie))

def train(importance=True, epochs=150, margin=0.5, track=False):
    torch.manual_seed(0); random.seed(1)
    conv = PinSageConv(); opt = torch.optim.Adam(conv.parameters(), lr=0.01); snaps = {}
    for ep in range(epochs):
        z = conv(feat, NB, AL, importance)
        q   = torch.randint(0, N, (96,))
        pos = torch.tensor([random.choice([j for j in range(N) if cluster[j] == cluster[qi] and j != qi]) for qi in q])
        neg = torch.tensor([random.choice([j for j in range(N) if cluster[j] != cluster[qi]]) for qi in q])
        sq, sp, sn = z[q], z[pos], z[neg]
        # J = max(0,  z_q . z_neg  -  z_q . z_pos  +  margin)
        loss = torch.clamp((sq * sn).sum(1) - (sq * sp).sum(1) + margin, min=0).mean()
        if track and ep in (0, 25, 50, 100, 149):
            zd = z.detach(); ia, ie = gap(zd)
            snaps[ep] = (round(ia - ie, 3), round(hit_rate(zd), 3), round(float(loss.detach()), 3))
        opt.zero_grad(); loss.backward(); opt.step()
    return z.detach(), snaps

z_imp, snaps = train(importance=True, track=True)
z_uni, _     = train(importance=False)
print("\ntraining curve  epoch -> (intra-inter cos gap, hit-rate, loss):")
for ep, s in snaps.items(): print(f"  ep {ep:3d}: gap={s[0]:+.3f}  hit={s[1]:.3f}  loss={s[2]:.3f}")
print("importance pooling: hit-rate", round(hit_rate(z_imp), 3), " intra/inter cos", tuple(round(x,3) for x in gap(z_imp)))
print("uniform-mean ablate: hit-rate", round(hit_rate(z_uni), 3), " intra/inter cos", tuple(round(x,3) for x in gap(z_uni)))

# Our small run (NOT the paper's numbers): with random-walk importance pooling + the max-margin ranking
# loss, training drives the loss to ~0 while the related-vs-unrelated similarity GAP widens
# (~0.24 -> ~0.93) and the hit-rate climbs (~0.67 -> 1.0) -- related items end up closer than negatives,
# exactly the qualitative effect PinSAGE is built to produce.

## Visualize the data & results

_As we train tiny item embeddings with random-walk importance pooling and the max-margin ranking loss, does the gap between related-item similarity and unrelated-item similarity actually widen — i.e. do related items end up closer than negatives — as the loss falls?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random
np.random.seed(0); random.seed(0); torch.manual_seed(0)

# Tiny synthetic item graph: 3 clusters of 7 related items + 5 misleading cross-cluster edges per item.
C, PER = 3, 7; N = C * PER
cluster = np.repeat(np.arange(C), PER)
adj = [[] for _ in range(N)]
for i in range(N):
    same  = [j for j in range(N) if j != i and cluster[j] == cluster[i]]
    other = [j for j in range(N) if cluster[j] != cluster[i]]
    adj[i] = same + random.sample(other, 5)

def visit_counts(start, n_walks=300, length=6):
    cnt = np.zeros(N)
    for _ in range(n_walks):
        cur = start
        for _ in range(length): cur = random.choice(adj[cur]); cnt[cur] += 1
    return cnt
T = 6; NB, AL = [], []
for u in range(N):
    cnt = visit_counts(u); order = np.argsort(-cnt)[:T]
    w = cnt[order].astype(np.float32); w = w / w.sum()
    NB.append(torch.tensor(order)); AL.append(torch.tensor(w))
feat = torch.randn(N, 8)
for c in range(C): feat[cluster == c] += torch.randn(8) * 1.0

class Conv(nn.Module):
    def __init__(s): super().__init__(); s.Q = nn.Linear(8, 16); s.W = nn.Linear(24, 16)
    def forward(s, z):
        out = []
        for u in range(z.shape[0]):
            msg = F.relu(s.Q(z[NB[u]])); n_u = (AL[u][:, None] * msg).sum(0)  # importance pooling
            h = F.relu(s.W(torch.cat([z[u], n_u]))); out.append(h / (h.norm() + 1e-9))
        return torch.stack(out)

def hit_rate(z):
    z = F.normalize(z, dim=1); sim = z @ z.t(); sim.fill_diagonal_(-9)
    return float(np.mean([cluster[sim[u].argmax().item()] == cluster[u] for u in range(N)]))
def gap(z):
    z = F.normalize(z, dim=1); ia, ie = [], []
    for i in range(N):
        for j in range(N):
            if i != j: (ia if cluster[i] == cluster[j] else ie).append(float((z[i]*z[j]).sum()))
    return np.mean(ia), np.mean(ie)

torch.manual_seed(0); random.seed(1)
conv = Conv(); opt = torch.optim.Adam(conv.parameters(), lr=0.01)
for ep in range(150):
    z = conv(feat)
    q   = torch.randint(0, N, (96,))
    pos = torch.tensor([random.choice([j for j in range(N) if cluster[j]==cluster[qi] and j!=qi]) for qi in q])
    neg = torch.tensor([random.choice([j for j in range(N) if cluster[j]!=cluster[qi]]) for qi in q])
    sq, sp, sn = z[q], z[pos], z[neg]
    loss = torch.clamp((sq*sn).sum(1) - (sq*sp).sum(1) + 0.5, min=0).mean()  # max-margin ranking loss
    if ep in (0, 25, 50, 100, 149):
        zd = z.detach(); ia, ie = gap(zd)
        print(f"ep {ep:3d}: gap={ia-ie:+.3f}  hit={hit_rate(zd):.3f}  loss={float(loss.detach()):.3f}")
    opt.zero_grad(); loss.backward(); opt.step()
# Our small run, not the paper's number: gap 0.235 -> 0.933, hit-rate 0.667 -> 1.0 as the loss -> 0.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Recompute an importance aggregation. Node $u$ has three neighbors with transformed messages
            $\mathbf{m}_1=(1,0)$, $\mathbf{m}_2=(0,2)$, $\mathbf{m}_3=(2,2)$ and random-walk visit counts
            $5$, $4$, $1$. Give the importance weights $\boldsymbol{\alpha}$ and the pooled vector $\mathbf{n}_u$,
            and compare with the uniform mean.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sum the visit counts: $5+4+1=10$. Normalize: $\boldsymbol{\alpha}=(0.5,\,0.4,\,0.1)$. — _Importance weights are the $L_1$-normalized visit counts (sum to $1$)._
- Weighted mean, coordinate 1: $0.5\cdot1+0.4\cdot0+0.1\cdot2=0.5+0+0.2=0.7$. — _Importance pooling multiplies each message by its $\alpha$ then sums._
- Coordinate 2: $0.5\cdot0+0.4\cdot2+0.1\cdot2=0+0.8+0.2=1.0$. So $\mathbf{n}_u=(0.7,\,1.0)$. — _Same weighted sum on the other coordinate._

**Answer:** $\boldsymbol{\alpha}=(0.5,0.4,0.1)$ gives $\mathbf{n}_u=(0.7,\,1.0)$. The uniform mean is
                 $\tfrac{1}{3}\big((1,0)+(0,2)+(2,2)\big)=\tfrac{1}{3}(3,4)=(1.0,\,1.333)$. Importance pooling
                 leans toward the most-visited neighbor $\mathbf{m}_1=(1,0)$ (weight $0.5$) and down-weights the
                 once-visited $\mathbf{m}_3=(2,2)$, so the two aggregates differ &mdash; exactly the effect the
                 random-walk weights are there to produce.

</details>

**Problem 2.** When does the margin loss vanish? Embeddings are unit vectors. For a triple, the
            query&ndash;positive dot product is $0.8$ and the query&ndash;negative dot product is $0.5$. With
            margin $\Delta=0.1$, is the loss zero? What is the smallest $\Delta$ that makes it positive?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Evaluate the inside of the $\max$: $\mathbf{z}_q\!\cdot\!\mathbf{z}_n-\mathbf{z}_q\!\cdot\!\mathbf{z}_i+\Delta = 0.5-0.8+0.1=-0.2$. — _Plug the dot products and the margin into Eqn. (1)._
- Since $-0.2\lt 0$, the hinge clips it: $\max\{0,-0.2\}=0$. — _The $\max\{0,\cdot\}$ makes already-satisfied triples cost nothing._
- Loss turns positive once $0.5-0.8+\Delta\gt 0$, i.e. $\Delta\gt 0.3$. — _The margin must exceed the current similarity gap $0.8-0.5=0.3$ to still penalize this triple._

**Answer:** With $\Delta=0.1$ the loss is $\max\{0,-0.2\}=0$ &mdash; the positive already beats the
                 negative by $0.3\gt\Delta$, so this triple is "done" and contributes no gradient. The loss only
                 becomes positive when $\Delta\gt 0.3$ (the size of the current gap). This is why a fixed margin
                 plus easy negatives lets the loss saturate to zero, and why the paper introduces
                 harder-and-harder negatives (&sect;3.3) to keep the inside of the $\max$ positive.

</details>

**Problem 3.** Ablation &mdash; remove the importance weighting. You replace PinSAGE's visit-count weighted
            mean with a plain uniform mean over the same top-$T$ neighbors, keeping everything else (the
            convolution shape, the max-margin loss) identical. On a graph whose top-$T$ neighbors are mostly the
            correct related items, do you expect a large or small change in hit-rate &mdash; and what would
            make the importance weighting matter more?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that if the top-$T$ neighborhood is already almost all relevant items, both the weighted and uniform aggregate point in nearly the same direction. — _When every neighbor is relevant, how you weight them barely changes the result._
- The weighting matters when the top-$T$ set is mixed: some relevant (high visit count), some noise (low count). — _Then the $L_1$-normalized counts suppress the noisy neighbors that a uniform mean would over-weight._
- Measure hit-rate and the related-vs-unrelated similarity gap for both aggregators; change only the aggregator. — _Isolating the aggregator attributes any difference to importance pooling &mdash; the exact part the paper introduces._

**Answer:** On an easy graph &mdash; where the random walk has already filtered the neighborhood down
                 to mostly-correct items &mdash; the change is small: both aggregators recover the clusters,
                 and our toy run shows comparable hit-rates. Importance pooling earns its keep when the top-$T$
                 set is noisy (popular but irrelevant neighbors leak in): there the visit-count weights
                 down-weight the noise that a uniform mean would treat as equally important. The CODEVIZ panel
                 reports the training-curve effect (margin gap and hit-rate rising as the max-margin loss falls);
                 the ablation print compares importance vs uniform pooling on the same toy graph.

</details>